# Exp 006 Soundscape Finetuning V2

## Goal

Build a stronger native soundscape training branch than `exp_004` while also making validation outputs fold-aware and reusable for later OOF analysis.

## Hypothesis

The native branch is already directionally correct, but it still underperforms on Kaggle because:
- the soundscape finetuning recipe is still too light
- `secondary_labels` are effectively unused
- single-fold validation is too sparse to trust small deltas

A second finetuning iteration with real masked-secondary weighting and fold-specific exports should give us a better native checkpoint and a much stronger evaluation base.

## What Changes Here

- keep the `exp_002` checkpoint as initialization
- keep soundscape background mixing
- turn on non-zero weighting for replay `secondary_labels`
- slightly expand the replay bank per class
- save every fold under its own output directory
- export best validation predictions so later notebooks can build OOF reports and postprocessing layers without rerunning training

## Expected Outcome

A stronger native soundscape checkpoint plus reusable fold artifacts for honest OOF-style comparison against `exp_003` and `exp_005`.


## Notebook Notes

- This notebook is the next native training iteration after the first `exp_005` Kaggle score of `0.737`.
- The main purpose is twofold:
  - improve the training recipe
  - improve the evaluation protocol
- Each fold writes to its own directory under `experiments/outputs/exp_006_soundscape_finetuning_v2/`.
- The best validation epoch also exports fold predictions for later native postprocessing and OOF analysis.


In [19]:
from __future__ import annotations

import ast
import json
import math
import random
import sys
from collections import OrderedDict
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from birdclef2026.reference_eval import (  # noqa: E402
    ReferenceModelConfig,
    build_reference_model_components,
    score_macro_auc,
)


@dataclass
class FinetuneConfig:
    seed: int = 42
    sample_rate: int = 32_000
    clip_duration: float = 5.0
    batch_size: int = 12
    num_workers: int = 0
    epochs: int = 6
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    use_amp: bool = True
    fold: int = 2
    n_folds: int = 5
    use_background_mixing: bool = True
    bg_mix_prob: float = 0.60
    bg_snr_db_min: float = 10.0
    bg_snr_db_max: float = 24.0
    bg_target_weight: float = 0.35
    use_train_audio_replay: bool = True
    max_replay_samples_per_class: int = 12
    replay_center_crop: bool = False
    secondary_weight: float = 0.20
    soundscape_cache_size: int = 24
    max_train_segments: int | None = None
    max_valid_segments: int | None = None
    save_every_epoch: bool = False
    export_validation_outputs: bool = True

    @property
    def clip_samples(self) -> int:
        return int(self.sample_rate * self.clip_duration)


CFG = FinetuneConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
TRAIN_AUDIO_DIR = DATA_DIR / 'train_audio'
TRAIN_SOUNDSCAPE_DIR = DATA_DIR / 'train_soundscapes'
EXP002_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_002_train_audio_reproduction'
BASE_OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_006_soundscape_finetuning_v2'
OUTPUT_DIR = BASE_OUTPUT_DIR / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
amp_enabled = CFG.use_amp and device.type == 'cuda'


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

{
    'project_root': str(PROJECT_ROOT),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'base_output_dir': str(BASE_OUTPUT_DIR),
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


{'project_root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026',
 'device': 'mps',
 'amp_enabled': False,
 'base_output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2',
 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2/fold_02',
 'seed': 42,
 'sample_rate': 32000,
 'clip_duration': 5.0,
 'batch_size': 12,
 'num_workers': 0,
 'epochs': 6,
 'learning_rate': 0.0003,
 'weight_decay': 0.0001,
 'use_amp': True,
 'fold': 2,
 'n_folds': 5,
 'use_background_mixing': True,
 'bg_mix_prob': 0.6,
 'bg_snr_db_min': 10.0,
 'bg_snr_db_max': 24.0,
 'bg_target_weight': 0.35,
 'use_train_audio_replay': True,
 'max_replay_samples_per_class': 12,
 'replay_center_crop': False,
 'secondary_weight': 0.2,
 'soundscape_cache_size': 24,
 'max_train_segments': None,
 'max_valid_segments': None,
 'save_every_epoch': False,
 'export_validation_outputs': True}

In [20]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
train_csv = pd.read_csv(DATA_DIR / 'train.csv')
species = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(species)}


def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    labels = sorted(set(label for item in series for label in parse_soundscape_labels(item)))
    return labels


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)
sc_clean['file_path'] = sc_clean['filename'].apply(lambda name: TRAIN_SOUNDSCAPE_DIR / name)
sc_clean = sc_clean[sc_clean['file_path'].map(Path.exists)].reset_index(drop=True)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].reset_index(drop=True)

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))
train_full_idx, valid_full_idx = full_splits[CFG.fold % len(full_splits)]
valid_files = set(full_df.iloc[valid_full_idx]['filename'].tolist())

train_df = (
    sc_clean[~sc_clean['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)
valid_df = (
    full_df[full_df['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)

if CFG.max_train_segments is not None:
    train_df = train_df.head(CFG.max_train_segments).reset_index(drop=True)
if CFG.max_valid_segments is not None:
    valid_df = valid_df.head(CFG.max_valid_segments).reset_index(drop=True)


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(species), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


train_targets = np.stack([encode_multi_hot(labels) for labels in train_df['label_list']], axis=0)
valid_targets = np.stack([encode_multi_hot(labels) for labels in valid_df['label_list']], axis=0)

{
    'fold': int(CFG.fold),
    'n_group_splits': int(len(full_splits)),
    'all_labeled_segments': int(len(sc_clean)),
    'fully_labeled_files': int(len(full_files)),
    'train_segments': int(len(train_df)),
    'valid_segments': int(len(valid_df)),
    'valid_files': int(len(valid_files)),
    'soundscape_classes_in_train': int((train_targets.sum(axis=0) > 0).sum()),
    'soundscape_classes_in_valid': int((valid_targets.sum(axis=0) > 0).sum()),
}


{'fold': 2,
 'n_group_splits': 5,
 'all_labeled_segments': 739,
 'fully_labeled_files': 59,
 'train_segments': 595,
 'valid_segments': 144,
 'valid_files': 12,
 'soundscape_classes_in_train': 73,
 'soundscape_classes_in_valid': 35}

In [21]:
def parse_secondary_labels(value: str) -> list[str]:
    if not isinstance(value, str) or value in ['', '[]']:
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except Exception:
        pass
    cleaned = value.strip('[]').replace("'", '').replace('"', '')
    return [item.strip() for item in cleaned.split(',') if item.strip()]


train_csv['file_path'] = train_csv.apply(
    lambda row: TRAIN_AUDIO_DIR / row['primary_label'] / row['filename'],
    axis=1,
)
replay_df = train_csv[
    train_csv['primary_label'].isin(label_to_idx)
    & train_csv['file_path'].map(Path.exists)
].copy()

if CFG.max_replay_samples_per_class is not None:
    replay_df = (
        replay_df
        .sort_values(['primary_label', 'rating'], ascending=[True, False])
        .groupby('primary_label', group_keys=False)
        .head(CFG.max_replay_samples_per_class)
        .reset_index(drop=True)
    )

{
    'use_train_audio_replay': CFG.use_train_audio_replay,
    'replay_samples': int(len(replay_df)),
    'replay_classes': int(replay_df['primary_label'].nunique()) if len(replay_df) else 0,
}


{'use_train_audio_replay': True, 'replay_samples': 0, 'replay_classes': 0}

In [22]:
class SoundscapeSegmentDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, bg_frame: pd.DataFrame, cfg: FinetuneConfig, label_to_idx: dict[str, int], is_train: bool):
        self.frame = frame.reset_index(drop=True)
        self.bg_frame = bg_frame.reset_index(drop=True) if len(bg_frame) else frame.reset_index(drop=True)
        self.cfg = cfg
        self.label_to_idx = label_to_idx
        self.is_train = is_train
        self.cache: OrderedDict[str, np.ndarray] = OrderedDict()

    def __len__(self) -> int:
        return len(self.frame)

    def _load_soundscape(self, file_path: Path) -> np.ndarray:
        key = str(file_path)
        if key in self.cache:
            audio = self.cache.pop(key)
            self.cache[key] = audio
            return audio

        audio, sr = sf.read(file_path, dtype='float32', always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != self.cfg.sample_rate:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.cfg.sample_rate)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        self.cache[key] = audio
        while len(self.cache) > self.cfg.soundscape_cache_size:
            self.cache.popitem(last=False)
        return audio

    def _slice_segment(self, audio: np.ndarray, start_sec: int) -> np.ndarray:
        start = int(start_sec * self.cfg.sample_rate)
        end = start + self.cfg.clip_samples
        clip = audio[start:end]
        if len(clip) < self.cfg.clip_samples:
            clip = np.pad(clip, (0, self.cfg.clip_samples - len(clip)))
        return clip.astype(np.float32)

    def _encode_labels(self, labels: list[str]) -> np.ndarray:
        target = np.zeros(len(self.label_to_idx), dtype=np.float32)
        for label in labels:
            idx = self.label_to_idx.get(label)
            if idx is not None:
                target[idx] = 1.0
        return target

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio = self._slice_segment(self._load_soundscape(Path(row['file_path'])), int(row['start_sec']))
        primary_target = self._encode_labels(row['label_list'])
        secondary_target = np.zeros_like(primary_target)

        bg_audio = np.zeros_like(audio)
        bg_target = np.zeros_like(primary_target)
        if self.is_train and self.cfg.use_background_mixing and len(self.bg_frame) > 1:
            bg_row = self.bg_frame.iloc[np.random.randint(0, len(self.bg_frame))]
            bg_audio = self._slice_segment(self._load_soundscape(Path(bg_row['file_path'])), int(bg_row['start_sec']))
            bg_target = self._encode_labels(bg_row['label_list'])

        site = '' if pd.isna(row['site']) else str(row['site'])
        return {
            'audio': torch.from_numpy(audio),
            'primary_target': torch.from_numpy(primary_target),
            'secondary_target': torch.from_numpy(secondary_target),
            'bg_audio': torch.from_numpy(bg_audio),
            'bg_target': torch.from_numpy(bg_target),
            'row_id': row['row_id'],
            'filename': row['filename'],
            'end_sec': int(row['end_sec']),
            'site': site,
            'hour_utc': int(row['hour_utc']),
            'source': 'soundscape',
        }


class TrainAudioReplayDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, cfg: FinetuneConfig, label_to_idx: dict[str, int]):
        self.frame = frame.reset_index(drop=True)
        self.cfg = cfg
        self.label_to_idx = label_to_idx

    def __len__(self) -> int:
        return len(self.frame)

    def _crop_or_pad(self, audio: np.ndarray) -> np.ndarray:
        target_len = self.cfg.clip_samples
        if len(audio) >= target_len:
            if self.cfg.replay_center_crop:
                start = max(0, (len(audio) - target_len) // 2)
            else:
                max_start = len(audio) - target_len
                start = 0 if max_start <= 0 else np.random.randint(0, max_start + 1)
            audio = audio[start : start + target_len]
        else:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio.astype(np.float32)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio, _ = librosa.load(row['file_path'], sr=self.cfg.sample_rate, mono=True)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
        audio = self._crop_or_pad(audio)

        primary_target = np.zeros(len(self.label_to_idx), dtype=np.float32)
        secondary_target = np.zeros(len(self.label_to_idx), dtype=np.float32)

        primary_idx = self.label_to_idx.get(row['primary_label'])
        if primary_idx is not None:
            primary_target[primary_idx] = 1.0

        for label in parse_secondary_labels(row.get('secondary_labels', '[]')):
            secondary_idx = self.label_to_idx.get(label)
            if secondary_idx is not None:
                secondary_target[secondary_idx] = 1.0

        return {
            'audio': torch.from_numpy(audio),
            'primary_target': torch.from_numpy(primary_target),
            'secondary_target': torch.from_numpy(secondary_target),
            'bg_audio': torch.zeros_like(torch.from_numpy(audio)),
            'bg_target': torch.zeros(len(self.label_to_idx), dtype=torch.float32),
            'row_id': Path(row['filename']).stem,
            'filename': row['filename'],
            'end_sec': -1,
            'site': '',
            'hour_utc': -1,
            'source': 'train_audio_replay',
        }


In [23]:
model_cfg = ReferenceModelConfig(num_classes=len(species))


class WaveformSEDClassifier(nn.Module):
    def __init__(self, model_cfg: ReferenceModelConfig):
        super().__init__()
        SEDModelCls, MelTransformCls = build_reference_model_components(model_cfg)
        self.mel = MelTransformCls()
        self.model = SEDModelCls()

    def forward(self, waveforms: torch.Tensor) -> dict[str, torch.Tensor]:
        mel = self.mel(waveforms)
        return self.model(mel)


class MaskedBCEWithSoftTargets(nn.Module):
    def __init__(self, secondary_weight: float = 0.0):
        super().__init__()
        self.secondary_weight = secondary_weight
        self.criterion = nn.BCEWithLogitsLoss(reduction='none')

    def forward(
        self,
        logits: torch.Tensor,
        primary_targets: torch.Tensor,
        secondary_targets: torch.Tensor,
        soft_targets: torch.Tensor | None = None,
    ) -> torch.Tensor:
        combined_targets = torch.clamp(primary_targets + secondary_targets, 0.0, 1.0)
        if soft_targets is not None:
            combined_targets = torch.maximum(combined_targets, soft_targets)

        raw_loss = self.criterion(logits, combined_targets)
        weights = torch.ones_like(raw_loss)
        secondary_only = (secondary_targets > 0) & (primary_targets <= 0)
        weights[secondary_only] = self.secondary_weight
        return (raw_loss * weights).sum() / weights.sum().clamp_min(1.0)


@torch.no_grad()
def summarize_macro_auc(y_true: np.ndarray, y_pred: np.ndarray, species: list[str]) -> dict:
    metrics = score_macro_auc(y_true, y_pred, species)
    return {
        'macro_auc': metrics['macro_auc'],
        'scored_classes': metrics['scored_classes'],
        'skipped_no_positive': metrics['skipped_no_positive'],
        'skipped_no_negative': metrics['skipped_no_negative'],
    }


def apply_background_mix(
    audio: torch.Tensor,
    bg_audio: torch.Tensor,
    bg_target: torch.Tensor,
    cfg: FinetuneConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    if not cfg.use_background_mixing:
        return audio, torch.zeros_like(bg_target)

    device = audio.device
    batch_size = audio.size(0)
    has_bg = (bg_target.sum(dim=1, keepdim=True) > 0).float()
    mix_mask = (torch.rand(batch_size, 1, device=device) < cfg.bg_mix_prob).float() * has_bg

    rms_audio = torch.sqrt(torch.mean(audio ** 2, dim=1, keepdim=True) + 1e-9)
    rms_bg = torch.sqrt(torch.mean(bg_audio ** 2, dim=1, keepdim=True) + 1e-9)
    snr_db = torch.empty(batch_size, 1, device=device).uniform_(cfg.bg_snr_db_min, cfg.bg_snr_db_max)
    scale = rms_audio / (rms_bg * (10.0 ** (snr_db / 20.0)) + 1e-9)

    mixed_audio = torch.clamp(audio + bg_audio * scale * mix_mask, -1.0, 1.0)
    soft_targets = torch.clamp(bg_target * scale * mix_mask * cfg.bg_target_weight, 0.0, 1.0)
    return mixed_audio, soft_targets


In [24]:
train_soundscape_dataset = SoundscapeSegmentDataset(train_df, train_df, CFG, label_to_idx, is_train=True)
valid_soundscape_dataset = SoundscapeSegmentDataset(valid_df, valid_df, CFG, label_to_idx, is_train=False)

train_dataset = train_soundscape_dataset
if CFG.use_train_audio_replay and len(replay_df):
    replay_dataset = TrainAudioReplayDataset(replay_df, CFG, label_to_idx)
    train_dataset = ConcatDataset([train_soundscape_dataset, replay_dataset])
else:
    replay_dataset = None

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=device.type == 'cuda',
    drop_last=False,
)
valid_loader = DataLoader(
    valid_soundscape_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=device.type == 'cuda',
    drop_last=False,
)

model = WaveformSEDClassifier(model_cfg).to(device)
criterion = MaskedBCEWithSoftTargets(secondary_weight=CFG.secondary_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))
scaler_device_type = 'cuda' if device.type == 'cuda' else 'cpu'
scaler = torch.amp.GradScaler(scaler_device_type, enabled=amp_enabled)

{
    'train_dataset_size': int(len(train_dataset)),
    'valid_dataset_size': int(len(valid_soundscape_dataset)),
    'replay_enabled': CFG.use_train_audio_replay,
    'replay_samples': 0 if replay_dataset is None else int(len(replay_dataset)),
    'train_batches': len(train_loader),
    'valid_batches': len(valid_loader),
    'parameters_millions': round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
}


{'train_dataset_size': 595,
 'valid_dataset_size': 144,
 'replay_enabled': True,
 'replay_samples': 0,
 'train_batches': 50,
 'valid_batches': 12,
 'parameters_millions': 6.25}

In [25]:
def save_checkpoint(path: Path, model, optimizer, scheduler, scaler, epoch: int, metrics: dict, cfg: FinetuneConfig, resume_mode: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'epoch': epoch,
        'metrics': metrics,
        'config': asdict(cfg),
        'resume_mode': resume_mode,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
    }
    torch.save(payload, path)


def move_optimizer_state(optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def load_experiment_checkpoint(path: Path, model, optimizer, scheduler, scaler, device: torch.device) -> dict:
    checkpoint = torch.load(path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    move_optimizer_state(optimizer, device)
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler_state = checkpoint.get('scaler_state_dict')
    if scaler_state:
        scaler.load_state_dict(scaler_state)
    return checkpoint


def load_pretrained_checkpoint(path: Path, model) -> dict:
    checkpoint = torch.load(path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    return checkpoint


def save_valid_artifacts(output_dir: Path, epoch: int, y_true: np.ndarray, y_pred: np.ndarray, meta_df: pd.DataFrame, species: list[str]) -> None:
    np.savez_compressed(
        output_dir / 'best_valid_outputs.npz',
        epoch=np.array([epoch], dtype=np.int32),
        y_true=y_true.astype(np.float32),
        y_pred=y_pred.astype(np.float32),
    )

    meta_df = meta_df.copy()
    meta_df['epoch'] = int(epoch)
    meta_df.to_csv(output_dir / 'best_valid_meta.csv', index=False)

    export_df = meta_df.copy()
    for idx, label in enumerate(species):
        export_df[f'true__{label}'] = y_true[:, idx].astype(np.int8)
        export_df[f'pred__{label}'] = y_pred[:, idx].astype(np.float32)

    try:
        export_df.to_parquet(output_dir / 'best_valid_predictions.parquet', index=False)
    except Exception:
        export_df.to_csv(output_dir / 'best_valid_predictions.csv', index=False)


def run_train_epoch(model, loader, optimizer, criterion, scaler, device, amp_enabled: bool, cfg: FinetuneConfig) -> float:
    model.train()
    loss_meter = 0.0
    sample_count = 0

    for batch in tqdm(loader, desc='train', leave=False):
        audio = batch['audio'].to(device, non_blocking=True)
        primary_target = batch['primary_target'].to(device, non_blocking=True)
        secondary_target = batch['secondary_target'].to(device, non_blocking=True)
        bg_audio = batch['bg_audio'].to(device, non_blocking=True)
        bg_target = batch['bg_target'].to(device, non_blocking=True)

        mixed_audio, soft_targets = apply_background_mix(audio, bg_audio, bg_target, cfg)

        optimizer.zero_grad(set_to_none=True)
        with (torch.amp.autocast(device_type=device.type, enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(mixed_audio)
            loss = criterion(output['clipwise_logit'], primary_target, secondary_target, soft_targets=soft_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = audio.size(0)
        loss_meter += loss.item() * batch_size
        sample_count += batch_size

    return loss_meter / max(1, sample_count)


@torch.no_grad()
def run_valid_epoch(model, loader, criterion, device, amp_enabled: bool, species: list[str]) -> dict:
    model.eval()
    loss_meter = 0.0
    sample_count = 0
    all_targets = []
    all_probs = []
    rows = []

    for batch in tqdm(loader, desc='valid', leave=False):
        audio = batch['audio'].to(device, non_blocking=True)
        primary_target = batch['primary_target'].to(device, non_blocking=True)
        secondary_target = batch['secondary_target'].to(device, non_blocking=True)

        with (torch.amp.autocast(device_type=device.type, enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(audio)
            loss = criterion(output['clipwise_logit'], primary_target, secondary_target, soft_targets=None)

        probs = torch.sigmoid(output['clipwise_logit']).detach().cpu().numpy()
        targets = primary_target.detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets)

        batch_size = audio.size(0)
        loss_meter += loss.item() * batch_size
        sample_count += batch_size

        rows.extend([
            {
                'row_id': row_id,
                'filename': filename,
                'end_sec': int(end_sec),
                'site': site,
                'hour_utc': int(hour_utc),
                'source': source,
            }
            for row_id, filename, end_sec, site, hour_utc, source in zip(
                batch['row_id'],
                batch['filename'],
                batch['end_sec'],
                batch['site'],
                batch['hour_utc'],
                batch['source'],
            )
        ])

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_probs, axis=0)
    metrics = summarize_macro_auc(y_true, y_pred, species)
    metrics['valid_loss'] = loss_meter / max(1, sample_count)
    meta_df = pd.DataFrame(rows)
    return {
        'metrics': metrics,
        'y_true': y_true,
        'y_pred': y_pred,
        'meta_df': meta_df,
    }


In [26]:
RUN_TRAINING = True
RESUME_TRAINING = True

history_path = OUTPUT_DIR / 'history.csv'
best_checkpoint_path = OUTPUT_DIR / 'best_model.pt'
last_checkpoint_path = OUTPUT_DIR / 'last_model.pt'
pretrained_checkpoint_path = EXP002_DIR / 'best_model.pt'
result_snapshot_path = OUTPUT_DIR / 'result_snapshot.json'

history: list[dict] = []
best_macro_auc = -math.inf
best_epoch = None
start_epoch = 1
resume_path = None
resume_mode = 'fresh'

if history_path.exists():
    history = pd.read_csv(history_path).to_dict('records')
    existing_macro = [float(row['macro_auc']) for row in history if pd.notna(row.get('macro_auc'))]
    if existing_macro:
        best_macro_auc = max(existing_macro)
        for row in history:
            if float(row.get('macro_auc', -math.inf)) == best_macro_auc:
                best_epoch = int(row['epoch'])

if RESUME_TRAINING:
    if last_checkpoint_path.exists():
        resume_path = last_checkpoint_path
    elif best_checkpoint_path.exists():
        resume_path = best_checkpoint_path

if resume_path is not None:
    checkpoint = load_experiment_checkpoint(resume_path, model, optimizer, scheduler, scaler, device)
    resume_epoch = int(checkpoint['epoch'])
    start_epoch = resume_epoch + 1
    checkpoint_macro_auc = float(checkpoint.get('metrics', {}).get('macro_auc', -math.inf))
    best_macro_auc = max(best_macro_auc, checkpoint_macro_auc)
    if checkpoint_macro_auc >= best_macro_auc:
        best_epoch = resume_epoch
    if history:
        history_df = pd.DataFrame(history)
        history = history_df[history_df['epoch'] <= resume_epoch].to_dict('records')
    resume_mode = 'resume_exp006'
elif pretrained_checkpoint_path.exists():
    pretrained_checkpoint = load_pretrained_checkpoint(pretrained_checkpoint_path, model)
    resume_mode = 'init_from_exp002'
else:
    raise FileNotFoundError(f'Missing exp_002 checkpoint: {pretrained_checkpoint_path}')

print({
    'resume_mode': resume_mode,
    'resume_path': None if resume_path is None else str(resume_path),
    'pretrained_checkpoint': str(pretrained_checkpoint_path) if pretrained_checkpoint_path.exists() else None,
    'start_epoch': start_epoch,
    'best_macro_auc': best_macro_auc,
    'best_epoch': best_epoch,
    'device': str(device),
})

if RUN_TRAINING:
    if start_epoch > CFG.epochs:
        print(
            f'Checkpoint already reached epoch {start_epoch - 1}. '
            f'Increase CFG.epochs above {start_epoch - 1} to continue training.'
        )
    else:
        for epoch in range(start_epoch, CFG.epochs + 1):
            train_loss = run_train_epoch(model, train_loader, optimizer, criterion, scaler, device, amp_enabled, CFG)
            valid_result = run_valid_epoch(model, valid_loader, criterion, device, amp_enabled, species)
            valid_metrics = valid_result['metrics']
            scheduler.step()

            row = {
                'epoch': epoch,
                'train_loss': train_loss,
                **valid_metrics,
                'learning_rate': optimizer.param_groups[0]['lr'],
            }
            history.append(row)
            history_df = pd.DataFrame(history)
            history_df.to_csv(history_path, index=False)

            print(row)

            save_checkpoint(last_checkpoint_path, model, optimizer, scheduler, scaler, epoch, row, CFG, resume_mode=resume_mode)

            if valid_metrics['macro_auc'] > best_macro_auc:
                best_macro_auc = valid_metrics['macro_auc']
                best_epoch = epoch
                save_checkpoint(best_checkpoint_path, model, optimizer, scheduler, scaler, epoch, row, CFG, resume_mode=resume_mode)
                if CFG.export_validation_outputs:
                    save_valid_artifacts(OUTPUT_DIR, epoch, valid_result['y_true'], valid_result['y_pred'], valid_result['meta_df'], species)

            if CFG.save_every_epoch:
                save_checkpoint(OUTPUT_DIR / f'epoch_{epoch:02d}.pt', model, optimizer, scheduler, scaler, epoch, row, CFG, resume_mode=resume_mode)
else:
    print('Training skipped. Set RUN_TRAINING = True to launch exp_006.')

if history_path.exists():
    history_df = pd.read_csv(history_path)
    if len(history_df):
        best_row = history_df.sort_values('macro_auc', ascending=False).iloc[0].to_dict()
        snapshot = {
            'experiment_id': 'exp_006',
            'experiment_name': 'soundscape_finetuning_v2',
            'fold': int(CFG.fold),
            'best_epoch': int(best_row['epoch']),
            'best_macro_auc': float(best_row['macro_auc']),
            'scored_classes': int(best_row['scored_classes']),
            'skipped_no_positive': int(best_row['skipped_no_positive']),
            'skipped_no_negative': int(best_row['skipped_no_negative']),
            'best_valid_loss': float(best_row['valid_loss']),
            'output_dir': str(OUTPUT_DIR),
            'secondary_weight': float(CFG.secondary_weight),
            'max_replay_samples_per_class': int(CFG.max_replay_samples_per_class),
        }
        result_snapshot_path.write_text(json.dumps(snapshot, indent=2))
        print('Snapshot:')
        print(json.dumps(snapshot, indent=2))


{'resume_mode': 'init_from_exp002', 'resume_path': None, 'pretrained_checkpoint': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_002_train_audio_reproduction/best_model.pt', 'start_epoch': 1, 'best_macro_auc': -inf, 'best_epoch': None, 'device': 'mps'}


train:   0%|          | 0/50 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [12, 313, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [7, 313, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero ele

valid:   0%|          | 0/12 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.09253682346404099, 'macro_auc': 0.7174993516840448, 'scored_classes': 35, 'skipped_no_positive': 199, 'skipped_no_negative': 0, 'valid_loss': 0.05352861310044924, 'learning_rate': 0.0002799038105676658}


...

train:   0%|          | 0/50 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [12, 313, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [7, 313, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero ele

valid:   0%|          | 0/12 [00:00<?, ?it/s]

{'epoch': 6, 'train_loss': 0.03263288670916016, 'macro_auc': 0.7724515414070539, 'scored_classes': 35, 'skipped_no_positive': 199, 'skipped_no_negative': 0, 'valid_loss': 0.045906429489453636, 'learning_rate': 0.0}
Snapshot:
{
  "experiment_id": "exp_006",
  "experiment_name": "soundscape_finetuning_v2",
  "fold": 2,
  "best_epoch": 6,
  "best_macro_auc": 0.7724515414070539,
  "scored_classes": 35,
  "skipped_no_positive": 199,
  "skipped_no_negative": 0,
  "best_valid_loss": 0.0459064294894536,
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2/fold_02",
  "secondary_weight": 0.2,
  "max_replay_samples_per_class": 12
}


/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_89316/2538932449.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'pred__{label}'] = y_pred[:, idx].astype(np.float32)
/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_89316/2538932449.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'true__{label}'] = y_true[:, idx].astype(np.int8)


In [27]:
if history_path.exists():
    history_df = pd.read_csv(history_path)
    history_df

    if result_snapshot_path.exists():
        print('\nSnapshot:')
        print(result_snapshot_path.read_text())

    print('\nOutput directory:', OUTPUT_DIR)
    print('Best checkpoint:', best_checkpoint_path.exists(), best_checkpoint_path)
    print('Validation export:', (OUTPUT_DIR / 'best_valid_outputs.npz').exists())
else:
    print('No history yet. Run the training cell first.')



Snapshot:
{
  "experiment_id": "exp_006",
  "experiment_name": "soundscape_finetuning_v2",
  "fold": 2,
  "best_epoch": 6,
  "best_macro_auc": 0.7724515414070539,
  "scored_classes": 35,
  "skipped_no_positive": 199,
  "skipped_no_negative": 0,
  "best_valid_loss": 0.0459064294894536,
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2/fold_02",
  "secondary_weight": 0.2,
  "max_replay_samples_per_class": 12
}

Output directory: /Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2/fold_02
Best checkpoint: True /Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_006_soundscape_finetuning_v2/fold_02/best_model.pt
Validation export: True


## Result Checklist

After each fold run, record the following in the project notes:

- fold id
- best validation `macro_auc`
- number of scored classes
- whether the non-zero `secondary_weight` looks stable or noisy
- whether exported validation predictions are present and readable

## Recommended Follow-Up

- run at least `2-3` folds before trusting small improvements over `exp_004`
- reuse the exported fold predictions for a stronger native OOF comparison against `exp_005`
- only after that decide whether the next gain should come from:
  - heavier postprocessing / stacker
  - stronger soundscape training
  - notebook-only pseudo-labeling
